In [3]:
#Faz a primeira requisição na API e prepara o JSON
import json

import requests

url = "https://pokeapi.co/api/v2/pokemon?limit=100&offset=0"

poke_api_json = requests.get(url).json()
    

In [ ]:
#Le todos os pokemons da API e grava na lista pokemons_data
pokemons_data = []

while poke_api_json["next"] is not None:
    pokemons_data.extend(poke_api_json["results"])

    response = requests.get(poke_api_json["next"])
    response.raise_for_status()

    poke_api_json = response.json()


In [ ]:
#Grava as informações de todos os pokemons no JSON
bronze_path = "../data/bronze/pokemon.json"

with open(bronze_path, "w") as file:
    json.dump(pokemons_data, file)

In [ ]:
import json

import pandas as pd

with open("../data/bronze/pokemon.json", "r", encoding="utf-8") as file_handle:
    pokemons_data = json.load(file_handle)

pokemon_df = pd.DataFrame(pokemons_data)

print(pokemon_df.head(10))

         name                                    url
0   bulbasaur   https://pokeapi.co/api/v2/pokemon/1/
1     ivysaur   https://pokeapi.co/api/v2/pokemon/2/
2    venusaur   https://pokeapi.co/api/v2/pokemon/3/
3  charmander   https://pokeapi.co/api/v2/pokemon/4/
4  charmeleon   https://pokeapi.co/api/v2/pokemon/5/
5   charizard   https://pokeapi.co/api/v2/pokemon/6/
6    squirtle   https://pokeapi.co/api/v2/pokemon/7/
7   wartortle   https://pokeapi.co/api/v2/pokemon/8/
8   blastoise   https://pokeapi.co/api/v2/pokemon/9/
9    caterpie  https://pokeapi.co/api/v2/pokemon/10/


In [ ]:
from pathlib import Path

import requests

BASE_DIR = Path.cwd().parent

for indice, pokemon in pokemon_df.iterrows():
    pokemon_path = (
    BASE_DIR 
        / "data" 
        / "bronze" 
        / "pokemons" 
        / f"{pokemon['name']}.json"
    )
    with open(pokemon_path, "w", encoding="utf-8") as file:
        pokemon_data = requests.get(pokemon['url']).json()
        json.dump(pokemon_data, file)

In [ ]:
#Verifica informações do parquet
from pathlib import Path

import pandas as pd

# Exibir todas as colunas
pd.set_option('display.max_columns', None)

# Exibir todas as linhas
pd.set_option('display.max_rows', None)

# Exibir o conteúdo completo das células (sem cortar textos longos)
pd.set_option('display.max_colwidth', None)

caminho_parquet = Path("../data/silver/pokemons/1.parquet")

df = pd.read_parquet(caminho_parquet)

teste_df = df.head()

df.head(5)

,id,name,base_experience,height,order,weight,stats,types,past_types
0,1,bulbasaur,64.0,7,1,69,"[{'base_stat': 45, 'effort': 0, 'stat': {'name': 'hp', 'url': 'https://pokeapi.co/api/v2/stat/1/'}}, {'base_stat': 49, 'effort': 0, 'stat': {'name': 'attack', 'url': 'https://pokeapi.co/api/v2/stat/2/'}}, {'base_stat': 49, 'effort': 0, 'stat': {'name': 'defense', 'url': 'https://pokeapi.co/api/v2/stat/3/'}}, {'base_stat': 65, 'effort': 1, 'stat': {'name': 'special-attack', 'url': 'https://pokeapi.co/api/v2/stat/4/'}}, {'base_stat': 65, 'effort': 0, 'stat': {'name': 'special-defense', 'url': 'https://pokeapi.co/api/v2/stat/5/'}}, {'base_stat': 45, 'effort': 0, 'stat': {'name': 'speed', 'url': 'https://pokeapi.co/api/v2/stat/6/'}}]","[{'slot': 1, 'type': {'name': 'grass', 'url': 'https://pokeapi.co/api/v2/type/12/'}}, {'slot': 2, 'type': {'name': 'poison', 'url': 'https://pokeapi.co/api/v2/type/4/'}}]",[]


In [ ]:
stats_list = teste_df['stats'].iloc[0]   # a lista de dicts

df_stats = pd.json_normalize(stats_list)
df_stats = df_stats.rename(columns={'stat.name': 'stat'})[['stat', 'base_stat']]
print(df_stats)


              stat  base_stat
0               hp         45
1           attack         49
2          defense         49
3   special-attack         65
4  special-defense         65
5            speed         45


In [ ]:
from pathlib import Path

import pandas as pd
from pokemonapi.transform import tratar_campos_summary

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 300)

new_df = []

i = 10
while i <= 20:
    caminho_parquet = Path(f"../data/silver/pokemons/{i}.parquet")
    df = pd.read_parquet(caminho_parquet)
    display(df)

    i += 1

new_df = tratar_campos_summary(df)

display(new_df)

In [ ]:
from pathlib import Path

import pandas as pd
from pokemonapi.transform import (
    criar_arquivo_summary_gold,
    summary_type_stats,
    tratar_campos_summary,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 30)

# summary_type_stats(new_df)
pokemon_df = []
i = 1
while i <= 1025:
    caminho_parquet = Path(f"../data/silver/pokemons/{i}.parquet")

    df = pd.read_parquet(caminho_parquet)
    df_summary = tratar_campos_summary(df)

    pokemon_df.append(df_summary)
    
    i += 1

lista_tratada = pd.concat(pokemon_df, ignore_index=True)
display(summary_type_stats(lista_tratada))

criar_arquivo_summary_gold(lista_tratada)



In [12]:
silver_path_summary = Path("../data/silver/clean_data/summary_pokemons.parquet")

summary_types = pd.read_parquet(silver_path_summary)

display(summary_types)

,id,name,base_experience,height,order,weight,hp,attack,defense,special_attack,special_defense,speed,tipo_1,tipo_2
0,1,bulbasaur,64.0,7,1,69,45.0,49.0,49.0,65.0,65.0,45.0,grass,poison
1,10,caterpie,39.0,3,14,29,45.0,30.0,35.0,20.0,20.0,45.0,bug,NaN
2,100,voltorb,66.0,5,158,104,40.0,30.0,50.0,55.0,55.0,100.0,electric,NaN
3,1000,gholdengo,275.0,12,977,300,87.0,60.0,95.0,133.0,91.0,84.0,steel,ghost
4,10001,deoxys-attack,270.0,17,515,608,50.0,180.0,20.0,180.0,20.0,150.0,psychic,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1346,995,iron-thorns,285.0,16,991,3030,100.0,134.0,110.0,70.0,84.0,72.0,rock,electric
1347,996,frigibax,64.0,5,949,170,65.0,75.0,45.0,35.0,45.0,55.0,dragon,ice
1348,997,arctibax,148.0,8,950,300,90.0,95.0,66.0,45.0,65.0,62.0,dragon,ice
1349,998,baxcalibur,300.0,21,951,2100,115.0,145.0,92.0,75.0,86.0,87.0,dragon,ice


In [ ]:
gold_path_types = Path("../data/gold/pokemons/type_summary.parquet")

df = pd.read_parquet(gold_path_types)

display(df)

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 60) 

SILVER_PATH = Path("../data/silver/clean_data")

df = pd.read_parquet(SILVER_PATH / 'summary_pokemons.parquet')

top_10 = (
    df.sort_values("attack", ascending=False)
      .head(10)
      [["id", "name", "attack", 'tipo_1', 'tipo_2']]
      .reset_index(drop=True)
)

display(top_10)
    

,id,name,attack,tipo_1,tipo_2
0,10043,mewtwo-mega-x,190.0,psychic,fighting
1,10047,heracross-mega,185.0,bug,fighting
2,798,kartana,181.0,grass,steel
3,10079,rayquaza-mega,180.0,dragon,flying
4,10001,deoxys-attack,180.0,psychic,NaN
5,10078,groudon-primal,180.0,ground,fire
6,10325,baxcalibur-mega,175.0,dragon,ice
7,10058,garchomp-mega,170.0,dragon,ground
8,10022,kyurem-black,170.0,dragon,ice
9,10157,necrozma-ultra,167.0,psychic,dragon


In [16]:
gold_path= Path("../data/gold/pokemons/top_10_weight.parquet")

summary_types = pd.read_parquet(gold_path)

display(summary_types)

,id,name,weight,tipo_1,tipo_2
0,10200,meowth-gmax,10000,normal,NaN
1,10219,toxtricity-amped-gmax,10000,electric,poison
2,10220,centiskorch-gmax,10000,fire,bug
3,10221,hatterene-gmax,10000,psychic,fairy
4,10222,grimmsnarl-gmax,10000,dark,fairy
5,10223,alcremie-gmax,10000,fairy,NaN
6,10224,copperajah-gmax,10000,steel,NaN
7,10225,duraludon-gmax,10000,steel,dragon
8,10226,urshifu-single-strike-gmax,10000,fighting,dark
9,10207,garbodor-gmax,10000,poison,NaN
